# Jonas Thamane Week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [ ]:
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI
import os


In [ ]:
load_dotenv(override=True)

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

PROVIDER = "openai"

In [ ]:
def get_client(provider: str):
    if provider == "openai":
        return OpenAI()
    
    elif provider == "ollama":
        return OpenAI(
            base_url="http://localhost:11434/v1",
            api_key="ollama"
        )
    
    else:
        raise ValueError("Unsupported provider")

In [ ]:
def build_messages(question: str):
    
    system_prompt = """
    You are an expert software engineer and technical educator.
    Your task is to:
    
    1. Explain the code clearly
    2. Break down each component
    3. Explain design reasoning
    4. Mention edge cases
    5. Suggest improvements
    6. Provide a short example if useful
    """
    
    user_prompt = f"""
    Please explain the following technical question in depth:
    
    {question}
    """
    
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]

In [ ]:
def ask_tech_tutor(question: str, provider="openai"):
    
    client = get_client(provider)
    model = MODEL_GPT if provider == "openai" else MODEL_LLAMA
    messages = build_messages(question)
    
    stream = client.chat.completions.create(
        model=model,
        messages=messages,
        stream=True
    )
    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            response += delta
            update_display(Markdown(response), display_id=display_handle.display_id)
    
    return response

In [ ]:
question = """
Explain what this Python code does and why:

yield from {book.get("author") for book in books if book.get("author")}
"""

ask_tech_tutor(question, provider=PROVIDER)